# Deliverable 2 — EDA, Data Leakage & Choosing the Right Model

**ADSC × American Airlines — Fall 2026**

In Deliverable 1 you were handed one model — `XGBRegressor` — and you trained it. It worked, and you got an R² somewhere north of 0.97.

Two questions were never asked:

1. **Was that score real?** (You cannot currently answer that. Section 3.)
2. **Was XGBoost the right choice?** (You have no idea, because you never tried anything else.)

This deliverable answers both. You will go deeper on EDA, learn what **data leakage** is and measure what it cost you, then run a genuine model-selection exercise: **Lasso vs Ridge**, **Random Forest vs XGBoost**, and **ARIMA** — plus the choice that sits underneath all of them and that nobody teaches you to make deliberately, your **loss function**.

---

## What you will learn

| Section | Topic | The question it answers |
|---|---|---|
| **2** | **Deeper EDA** | What does the data actually look like, and what does that imply about which model to reach for? |
| **3** | **Data leakage** | Why was the Deliverable 1 score too good, and how do I stop it happening again? |
| **4** | **Lasso vs Ridge** | Two linear models, two penalties. When do I want shrinkage, and when do I want feature selection? |
| **5** | **Loss functions** | MSE, MAE, Huber, quantile — what am I actually asking the model to minimise, and does it match what the business cares about? |
| **6** | **XGBoost vs Random Forest vs ARIMA** | Three model families. Which fits *this* problem, and why is one of them answering a completely different question? |
| **7** | **Pick one and defend it** | Given everything above, what would you ship to American Airlines? |

The single most important idea in here: **model selection is not a leaderboard.** It is a set of choices — model family, loss function, regularisation — each of which should follow from something you observed in the data or something the business told you. By the end you should be able to justify every one of your choices without using the phrase "it scored the best".

---

## What we are building on from Deliverable 1

We assume all of this and will not re-explain it:

- loading the CSV, `.shape` / `.info()` / `.describe()`
- null counts, mean / median / std / quartiles / IQR with NumPy
- the 1.5 × IQR outlier rule
- the price histogram, price-by-airline box plot, duration-vs-price scatter
- `LabelEncoder`, an 80/20 `train_test_split`, `XGBRegressor`
- MAE, RMSE, R², and gain-based feature importance

Keep **your own completed Deliverable 1 notebook** open in another tab — it is in `deliverable1/`, saved under your name — because we refer back to it by section number throughout.

---

## The git workflow changed — read this before you touch anything

Deliverable 1 was solo: you cut `deliverable1/<your-name>` and opened a PR straight into `main`. With 70+ members that does not scale, and it is not how real teams work either.

**From Deliverable 2 onward you work in a team, and your team owns a branch tree.**

```
main                          <- protected. Only a PM merges here.
 └── team-06/main             <- YOUR TEAM's integration branch ("team trunk")
      ├── team-06/alex-chen    <- a member branch: your own work
      ├── team-06/priya-r      <- another member, in parallel
      └── team-06/sam-lee
```

**What matters is the structure, not the names.** Three rules:

1. **Your team has one trunk branch.** The suggested name is `team-##/main` — your team number, zero-padded, so team 6 is `team-06`. If that exact name is unavailable (see the note below), any name your team agrees on is completely fine.
2. **You personally work on your own branch off that trunk**, e.g. `team-##/<your-name>`. Never commit directly to the trunk.
3. **Work moves up the tree:** your branch → your team's trunk → `main`. Your PR goes into **the trunk**, reviewed by a teammate. Only the trunk opens a PR into `main`, and only a PM merges that one.

### Setting it up

**One person on the team** creates the trunk (do this once, then tell your teammates it exists):

```sh
git switch main
git pull origin main
git switch -c team-06/main          # use YOUR team number
git push -u origin team-06/main
```

**Every member** then branches off the team trunk — not off `main`:

```sh
git fetch origin
git switch -c team-06/alex-chen origin/team-06/main
git push -u origin team-06/alex-chen
```

### Submitting

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push

# then on GitHub: open a PR from team-06/alex-chen  ->  team-06/main
# (base branch is your TEAM's trunk, NOT the repo's main)
```

> **⚠️ If `git switch -c team-##/main` fails, this is why.** Git stores branch names as file paths, so a branch name cannot be a *prefix* of another. If a plain branch named `team-07` already exists, then `team-07/main` is **impossible** — git says `cannot lock ref ... 'team-07' exists`. Several teams are in exactly this position from earlier in the term.
>
> **This is not a problem, and nothing needs deleting.** Just pick a name that is free and tell your team what it is:
>
> ```sh
> git switch -c team-07-main      # a dash instead of a slash -- always works
> ```
>
> The same rule applies to member branches: `team-06/alex-chen` and `team-06/alex-chen-v2` are fine, but `team-06/alex-chen/fix` would be blocked by `team-06/alex-chen`. Keep each name a flat leaf.

> **⚠️ Notebooks and merge conflicts.** A `.ipynb` is JSON, and git conflicts on it are genuinely horrible. Two habits make it survivable:
> - **Clear outputs before every commit** (Kernel → Restart & Clear Output). Most notebook conflicts are rendered output, not code.
> - **Each member completes their own copy of this notebook.** You are *not* meant to split the cells across teammates. The notebook is individual learning; the shared team artifact is `deliverable2/TEAM_REPORT.md`, which your team fills in together on the team trunk.

Full detail, including each member's named review duty, is in [`deliverable2/README.md`](README.md). The general workflow rules have not changed — [`CONTRIBUTING.md`](../CONTRIBUTING.md) is still the operating manual.

---

> ### 🗒️ A note on AI tools
>
> Half of this deliverable is *judgment*: is this leakage, is Lasso the right penalty, does ARIMA even apply here. An AI will hand you a confident answer to every one of those, and you will have learned nothing and be unable to defend it when a teammate asks in review.
>
> **Use AI as a tutor, not an author:**
> - "Why does Lasso drive coefficients to exactly zero when Ridge does not?"
> - "What does `objective='reg:pseudohubererror'` change about training?"
> - "Why is ARIMA inappropriate for data without a time index?"
>
> **You must be able to explain every line you submit.** A teammate *will* review your PR and *will* ask.

---

## How to approach this deliverable

Read this before you start. It will save you more time than it costs.

### 1. The written answers are the deliverable — not the plots

Every section ends with a markdown prompt asking *why*. Those are the thing being assessed. A plot with no conclusion attached is worth nothing, and a number with no interpretation is worth less than no number at all, because it invites someone to act on it.

Concretely: when a cell tells you to make a histogram, the deliverable is not the histogram. It is the sentence **"...therefore I should \_\_\_ when I model."** If you cannot finish that sentence, you have not finished the cell.

### 2. The honest score is the goal, whichever direction it moves

Section 3 removes three leaks from your Deliverable 1 setup and measures what each was worth. **Do not assume the number must fall.** On some datasets a leak is worth a lot of fake R² and the honest score drops sharply; on others it turns out to have been adding almost nothing, and your score barely moves or even ticks up.

Both outcomes are real results. **The point is not that your score goes down — it is that you can no longer tell which it will be until you remove the leaks and measure.** An unmeasured score is untrustworthy no matter how good it looks.

So: report whatever you get, and never tune your way toward a number you were hoping for. If your honest R² does drop, being able to explain why the lower number is the real one is the single most professional thing a data scientist does. If it does not drop, say so and explain why that leak mattered less here than it might have.

### 3. How to think about choosing a model

The instinct from Deliverable 1 is "run them all, keep the best score". Resist it. Work through the questions in this order, because each one can eliminate options before you spend compute on them:

1. **What shape is my data, and what shape does this model require?** This is first because it is disqualifying. ARIMA needs an ordered time series; if you do not have one, no amount of tuning helps. Section 6 is built around this question.
2. **What am I asking the model to minimise?** The default is squared error, which is a choice you have been making silently. Section 5 is about making it on purpose.
3. **What does the stakeholder need?** A client who needs to *explain* a price to a customer may be better served by a Ridge model with readable coefficients than by a boosted ensemble that scores two points higher. Accuracy is one requirement among several.
4. **What does it cost?** Fit time, tuning burden, how many ways it can break in production. A model that takes 40× longer for 1% more R² is usually the wrong call.
5. **Only then: what scores best?**

If your final justification is "it had the highest R²", you have skipped the deliverable.

### 4. How to work the code

- **Run cells in order, top to bottom.** Later sections depend on variables defined earlier (`clean`, `X_train`, `prep_tree`). If something is `None` or undefined, you skipped a cell.
- **Check your work as you go.** After any transform, look at the result — `.head()`, `.shape`, `.value_counts()`. Most bugs in data code are silent: the wrong answer computes perfectly happily.
- **Some cells are slow.** Random Forest on 240k rows and 5-fold CV both take minutes. If a cell is painful on your machine, **subsample it and say so in your PR** — that is a normal engineering decision, not cheating. `X.sample(80_000, random_state=RANDOM_STATE)` is your friend.
- **Read the error, then the line above it.** Python tracebacks name the failing line. Nine times in ten the actual mistake is one line earlier.
- **Commit per section**, not in one lump at the end. Small commits are the entire reason your history is useful to a reviewer.

### 5. How to work with your team

Each member completes **their own full copy** of this notebook — the work is individual, because the learning is individual. What your team does **together** is agree on the conclusions and write them into `TEAM_REPORT.md` on your team's trunk branch.

Beyond that, how your team organises itself is up to your team. Decide in your first meeting and write the decision down somewhere your PM can see it.

One piece of hard-won advice: **clear your outputs before every commit** (Kernel → Restart & Clear Output). Notebook merge conflicts are mostly conflicts in rendered output, not in code, and this one habit prevents the large majority of them.

---

## References

Every link below was checked and goes to the specific page you need — not a search result or a landing page. Keep the ones for your current section open in browser tabs.

### Start here

| | |
|---|---|
| **Dataset** — Flight Price Prediction (EaseMyTrip) | https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction |
| **scikit-learn: Common pitfalls and recommended practices** — the single most useful page for this deliverable. Shows a "Wrong" and a "Right" version of each mistake. | https://scikit-learn.org/stable/common_pitfalls.html |
| **An Introduction to Statistical Learning** — the standard undergraduate text, **free PDF**. Ch. 6 covers Ridge/Lasso, Ch. 8 covers trees and boosting. Python edition (ISLP) also available. | https://www.statlearning.com/ |

### Section 3 — leakage & validation

| | |
|---|---|
| Data leakage, with a worked example of a score collapsing from 0.76 to chance once the leak is fixed | https://scikit-learn.org/stable/common_pitfalls.html#data-leakage |
| Inconsistent preprocessing — why the transform must be fitted once and applied, not refitted | https://scikit-learn.org/stable/common_pitfalls.html#inconsistent-preprocessing |
| Pipelines and composite estimators (`Pipeline`, `ColumnTransformer`) | https://scikit-learn.org/stable/modules/compose.html |
| Cross-validation: evaluating estimator performance | https://scikit-learn.org/stable/modules/cross_validation.html |

### Section 4 — Lasso vs Ridge

| | |
|---|---|
| Ridge regression — §1.1.2 of the Linear Models guide | https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification |
| Lasso — §1.1.3, including the note on why the objective is scaled by `1/(2·n_samples)` | https://scikit-learn.org/stable/modules/linear_model.html#lasso |
| Elastic-Net — §1.1.5 | https://scikit-learn.org/stable/modules/linear_model.html#elastic-net |
| **Common pitfalls in interpreting linear coefficients** — worked example of why scale matters, why coefficients are conditional not marginal, and how correlated features make coefficients unstable. Directly relevant to reading your Lasso output. | https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html |

### Section 5 — loss functions

| | |
|---|---|
| XGBoost learning-task parameters — the full `objective` list (`reg:squarederror`, `reg:absoluteerror`, `reg:pseudohubererror`, `reg:quantileerror`) | https://xgboost.readthedocs.io/en/stable/parameter.html#learning-task-parameters |
| **Prediction intervals for gradient boosting regression** — the reference implementation for Section 5's quantile models, including the coverage check. Note its own conclusion: *each model performs best on the metric it was trained for.* | https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_quantile.html |
| Regression metrics — MAE, RMSE, R², and `mean_pinball_loss` | https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics |

### Section 6 — model families

| | |
|---|---|
| **Introduction to Boosted Trees** — the clearest explanation of additive training anywhere, and it explicitly notes that random forests and boosted trees share the same model form and differ in *how they are trained*. Read this before writing your bagging-vs-boosting answer. | https://xgboost.readthedocs.io/en/stable/tutorials/model.html |
| Random forests and randomized tree ensembles — §1.11.2 | https://scikit-learn.org/stable/modules/ensemble.html#random-forests |
| Gradient-boosted trees — §1.11.1 | https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees |
| `statsmodels` ARIMA API — the `order=(p, d, q)` signature | https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html |
| **Forecasting: Principles and Practice, Ch. 9 — ARIMA models.** Free online, and the standard reference. | https://otexts.com/fpp3/arima.html |
| §9.1 Stationarity and differencing — read this before choosing your `d` | https://otexts.com/fpp3/arima.html#sec-stationarity |

---

## Setup

Copy the dataset you already downloaded to **`deliverable2/data/Clean_Dataset.csv`**. That path is git-ignored; **do not commit the CSV.**

One new package this time — `statsmodels`, for ARIMA:

```sh
pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels
```

---
# Section 0 — Imports & helpers

Run this cell first. Everything below depends on it.

Some cells in this notebook are **given to you complete** — the helpers, and the reproduction of your Deliverable 1 baseline. Read them; they are the reference implementations you will copy patterns from. The cells marked `# TODO` are yours.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- splitting & validation ------------------------------------------------
from sklearn.model_selection import train_test_split, KFold, cross_validate

# --- preprocessing ---------------------------------------------------------
# OneHotEncoder  : one 0/1 column per category. No invented ordering.
# OrdinalEncoder : integers in an order YOU specify (zero < one < two_or_more).
# StandardScaler : centres and scales numerics. Optional for trees,
#                  MANDATORY for Lasso/Ridge -- Section 4 explains why.
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# --- composition -----------------------------------------------------------
# Pipeline guarantees every transform is fitted on the TRAINING fold only.
# That is our main defence against the leakage in Section 3.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- the models we will compare -------------------------------------------
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# --- metrics ---------------------------------------------------------------
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_pinball_loss)

# --- display ---------------------------------------------------------------
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

from pathlib import Path
from sklearn.base import clone
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42   # every split, every model. Reproducibility is not optional.
DATA_PATH = 'deliverable2/data/Clean_Dataset.csv'   # relative to the repo root

if not Path(DATA_PATH).exists():
    DATA_PATH = 'data/Clean_Dataset.csv'

print('pandas', pd.__version__, '| numpy', np.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)
import xgboost; print('xgboost', xgboost.__version__)

# statsmodels is only needed in Section 6c (ARIMA). If this import fails,
# `pip install statsmodels` and restart the kernel.
import statsmodels.api as sm
print('statsmodels', sm.__version__)

In [ ]:
def regression_report(y_true, y_pred, label=''):
    """Return a dict of the four metrics we use everywhere in this notebook.

    MAE   -- average absolute miss, in rupees. Interpretable.
    RMSE  -- squared-error based, so large misses dominate. RMSE >> MAE means a
             few very bad predictions are hiding behind a decent average.
    R2    -- fraction of price variance explained.
    MAPE% -- average miss as a PERCENTAGE of true price. The business metric: being
             off by 2,000 INR is trivial on a 52,000 ticket and awful on a 6,000 one.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'model': label,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': r2_score(y_true, y_pred),
        'MAPE%': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
    }


# Running scoreboard. Every model we fit goes on here, so at the end we can read
# the whole deliverable as one comparison table.
SCOREBOARD = []

def record(result):
    """Append a regression_report dict to the scoreboard and show the table."""
    SCOREBOARD.append(result)
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)


def show_scoreboard():
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)

print('Helpers defined: regression_report, record, show_scoreboard')

---
# Section 1 — Load the data

Nothing new here conceptually — but note the two given lines after the load, and read the comment attached to them.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(raw.shape, raw.columns.tolist())

In [ ]:
# Two housekeeping fixes, given to you.
#
# 1. The CSV as published ships a nameless first column that pandas labels
#    'Unnamed: 0'. It is the row number from whoever exported the file -- somebody
#    ran to_csv() without index=False. It is NOT a feature: a flight you are asked
#    to price next week has no row number in your training file. Drop it.
#    (Deliverable 1 never dropped it, which means it was sitting in your X. Section 3
#    measures what that cost you.)
#
# 2. The Deliverable 1 schema table used 'Source City' / 'Price'; the actual CSV
#    uses 'source_city' / 'price'. Documentation drifting from the file is routine.
#    We lowercase everything so the rest of the notebook works either way.

df = raw.copy()
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df = df.drop(columns=[c for c in df.columns if c.startswith('unnamed')])

print('shape:', df.shape)
print('columns:', df.columns.tolist())
df.head(3)

---
# Section 2 — EDA that informs a modelling decision

**Goal:** go deeper than Deliverable 1, and tie every observation to a modelling consequence.

Deliverable 1's EDA was descriptive: here is the distribution, here are the categories, here is a scatter plot. Useful, but it stopped short. **EDA earns its keep when it changes what you do next.** So for every plot in this section, the deliverable is not the plot — it is the sentence "...therefore I should \_\_\_ when I model".

Four things we need to know before choosing a model:

| # | What we look at | The modelling decision it drives |
|---|---|---|
| 2a | How `price` is distributed | Which **loss function** (Section 5) and whether to transform the target |
| 2b | Whether relationships are **linear** | Whether a linear model (Section 4) can compete at all |
| 2c | The **booking curve** over `days_left` | Whether a **time-series** model (Section 6) has anything to work with |
| 2d | Category counts and dominance | Which **encoding**, and whether one feature will swamp everything |

---

### 2a — The shape of the target

Deliverable 1 plotted the price histogram and noted it was right-skewed. Go further: plot it **split by class**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, log_scale in zip(axes, [False, True]):
    sns.histplot(data=df, x='price', hue='class', bins=60, log_scale=log_scale, ax=ax)
    ax.set_xlabel('Fare (INR, log scale)' if log_scale else 'Fare (INR)')
print(df.groupby('class').price.agg(['count', 'mean', 'median', 'std']))
print('Skew, raw and log:', df.price.skew(), np.log1p(df.price).skew())

**Price by cabin.** Mean economy fare was ₹6,572 and mean business fare was ₹52,540. Log transformation reduced skew from 1.061 to 0.405. A ₹5,000 miss means very different things in these two cabins, so report MAE in rupees alongside percentage error.

### 2b — Is this problem linear?

Before you can argue that you need a boosted tree ensemble, you have to know what a straight line gets you. And before you can read a correlation matrix, note the limit of the tool: **Pearson correlation measures only the *linear* part of a relationship.** A perfect U-shaped relationship has a correlation near zero. A low correlation does not mean "no relationship" — it means "no *linear* relationship", which is a much narrower claim.

In [ ]:
numeric = df[['duration', 'days_left', 'price']].assign(is_business=df['class'].eq('Business').astype(int))
sns.heatmap(numeric.corr(), annot=True, center=0, cmap='coolwarm')
plt.figure()
sns.regplot(data=df.sample(5000, random_state=42), x='duration', y='price', ci=None, scatter_kws={'s': 4, 'alpha': .3})

**Linear signal.** Cabin had the largest observed correlation with price (0.938), followed by duration (0.204) and lead time (−0.092 in signed terms). A straight line through duration averages over two cabin price bands. Duration can still matter within cabins or through interactions with route and stops.

### 2c — The booking curve: the one piece of this dataset with a time axis

This is the most important plot in the notebook, and it is the one Deliverable 1 never made.

`days_left` is **days until departure**. It is not a timestamp, but it *is* an ordered horizon — so if you average price at each value of `days_left`, you get a genuine **sequence**: how fare moves as departure approaches. Airlines call this the booking curve, and forecasting it is a real job.

That ordered sequence is the only thing in this dataset a time-series model like ARIMA can operate on. Section 6c comes back to it.

In [ ]:
curve = df.groupby('days_left').price.agg(['mean', 'median', 'count'])
cabins = df.pivot_table(index='days_left', columns='class', values='price')
ax = cabins.join(curve['mean'].rename('All')).plot(ylabel='Mean fare (INR)')
ax.invert_xaxis()
print(len(curve), 'lead times')
print(cabins.loc[[1, 2, 7, 14, 30, 49]])

**Booking curve.** Mean economy fare rose from ₹5,339 at 30 days to ₹9,455 at 14 days and ₹14,613 at one day. Business fares also rose near departure, from ₹52,057 at 30 days to ₹65,169 at one day. The proportional increase was larger in economy. A spline or lead-time bins would let a linear model change slope. Only 49 distinct lead times are available for the aggregate series.

### 2d — Categories: counts, dominance, and encoding

In [ ]:
for name in df.select_dtypes(exclude='number').columns:
    values = df[name].value_counts()
    print(name, len(values), values.head(8).to_dict())
print(pd.crosstab(df.airline, df['class'], values=df.price, aggfunc='mean'))
routes = df.groupby(['source_city', 'destination_city']).price.mean().sort_values()
print(routes.head(), routes.tail())

> **Which columns are ordered, and which are not?** This matters in Section 4 more than you would guess.
>
> - `stops` is **ordinal**: `zero < one < two_or_more`. That is a real ranking.
> - `departure_time` / `arrival_time` are **ordinal**: `Early_Morning < Morning < Afternoon < Evening < Night < Late_Night` — they are a clock.
> - `airline`, `source_city`, `destination_city` are **nominal**. There is no sense in which Vistara > SpiceJet.
>
> Deliverable 1 used `LabelEncoder` on all seven, which assigns integers **alphabetically**. For `stops` that gives `one=0, two_or_more=1, zero=2` — telling the model a direct flight ranks *above* a two-stop flight. Not just arbitrary: backwards.
>
> A tree can route around this by spending extra splits to separate the categories again, so the cost shows up as wasted capacity. **A linear model cannot route around it at all** — it gets one coefficient per column and is forced to assume evenly-spaced, ordered effects. Section 4 shows you exactly how much that costs.

### 2e — EDA decisions

| Observation | Modeling decision |
|---|---|
| Business fares average ₹52,540 and economy ₹6,572. | Retain cabin and inspect percentage errors as well as rupee errors. |
| The cabin indicator correlates 0.938 with fare. | Include cabin explicitly in the preprocessing coverage check. |
| The booking curve steepens close to departure, particularly in economy relative to its usual price. | Allow a nonlinear lead-time effect and interactions with cabin. |
| Stops and time bands have an order. Cities and airlines are labels. | Encode stops and time bands in their observed order and one-hot encode nominal fields. Clock time wraps at midnight, a limitation of the ordinal encoding. |

---
# Section 3 — Data leakage

**Goal:** find the three places the Deliverable 1 setup let information leak, fix each one, and **measure what each was actually worth** rather than assuming.

> **Leakage** is when information reaches the model at training time that will not be available at prediction time.

It is not exotic. It is the default result of writing code in the order that feels natural — "clean everything, then split" — because every transform fitted before the split has already seen the test set.

Deliverable 1 had three leaks:

| # | Leak | Where |
|---|---|---|
| 1 | **The row counter.** `Unnamed: 0` was never dropped, so it sat in `X`. A future flight has no row number. | D1 §3a |
| 2 | **Duplicate rows across the split.** Once the unique counter is removed, this dataset contains repeated rows; copies can land in train *and* test, so the model is graded on rows it memorised. | D1 §3c |
| 3 | **Encoders fitted on the full dataset.** `df[col] = le.fit_transform(df[col])` ran in §3b, *before* the split in §3c. | D1 §3b |

**Be precise about leak 3's severity**, because overstating it is its own error. A plain `LabelEncoder` on complete categorical columns learns a list of category *names* — it learns nothing about price, so the practical damage is small. It is still **wrong as a habit**, because the identical code shape is severe the moment the transform learns something numeric:

- a `StandardScaler` fitted pre-split leaks the test set's mean and variance
- a **target encoder** fitted pre-split leaks the test set's *prices* directly
- an imputer fitted pre-split leaks the test distribution

You will not remember which transforms are safe at 2am. **Fit every transform inside the split, always.** That is what `Pipeline` is for.

### 3a — Reproduce the Deliverable 1 baseline (given)

Given to you complete, so every member starts from the identical reference number. Read it and match each line to a section of the old notebook.

In [ ]:
def deliverable1_baseline(path, random_state=RANDOM_STATE, remove_counter=False, deduplicate=False):
    """Reproduce Deliverable 1 exactly -- leaks and all.

      D1 S1  : read_csv
      D1 S2a : dropna
      D1 S3a : drop 'flight'        <- but NOT the row counter
      D1 S3b : LabelEncoder on the FULL frame, before the split   <- leak 3
      D1 S3c : X = everything except price; 80/20 split           <- dupes straddle it
      D1 S4  : XGBRegressor, same hyperparameters
    """
    from sklearn.preprocessing import LabelEncoder

    frame = pd.read_csv(path)
    frame.columns = [c.strip().lower().replace(' ', '_') for c in frame.columns]
    frame = frame.dropna().drop(columns=['flight'])

    if remove_counter:
        frame = frame.drop(columns=frame.filter(regex="^unnamed").columns)
    if deduplicate:
        frame = frame.drop_duplicates()

    for col in ['airline', 'source_city', 'departure_time', 'stops',
                'arrival_time', 'destination_city', 'class']:
        # Fitted on 100% of rows -- including the ones we will call "unseen".
        frame[col] = LabelEncoder().fit_transform(frame[col])

    X = frame.drop(columns=['price'])
    y = frame['price']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state)

    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=random_state, n_jobs=4)
    model.fit(X_tr, y_tr)
    return model, X_tr, X_te, y_tr, y_te, model.predict(X_te)


t0 = time.time()
d1_model, d1_X_tr, d1_X_te, d1_y_tr, d1_y_te, d1_pred = deliverable1_baseline(DATA_PATH)
print(f'trained in {time.time() - t0:.1f}s')
print('features the D1 model was given:', d1_X_tr.columns.tolist())

record(regression_report(d1_y_te, d1_pred, 'D1 baseline (3 leaks)'))

In [ ]:
print(pd.Series(d1_model.feature_importances_, index=d1_X_tr.columns).sort_values(ascending=False))
print(raw.groupby('class')[raw.columns[0]].agg(['min', 'max']))

### 3b — Remove the leaks, one at a time

One change per step, so each result is **attributable**. Change three things at once and you learn nothing about any of them.

In [ ]:
_, _, _, _, observed, predicted = deliverable1_baseline(DATA_PATH, remove_counter=True)
record(regression_report(observed, predicted, 'No counter'))

In [ ]:
_, _, _, _, observed, predicted = deliverable1_baseline(DATA_PATH, remove_counter=True, deduplicate=True)
record(regression_report(observed, predicted, 'Deduplicated'))

### 3c — Step 3: a `Pipeline`, so leakage becomes structurally impossible

A **`Pipeline`** is a list of `(name, transform)` steps ending in a model, wrapped to behave as one estimator. What matters is what `.fit()` does:

```
pipe.fit(X_train, y_train)
   -> every transform learns ONLY from X_train
pipe.predict(X_test)
   -> every transform APPLIES what it learned; it never re-learns from X_test
```

You cannot accidentally fit on the test set, because you never call `.fit` on a transform yourself. And this becomes essential rather than merely tidy in Section 4, where we cross-validate over many `alpha` values: without a pipeline, the scaler would be refitted on each fold's test data and every number would be slightly wrong.

A **`ColumnTransformer`** routes different columns to different transforms:

```python
ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
    ('ord', OrdinalEncoder(categories=[...]),       ordinal_cols),
    ('num', StandardScaler(),                       numeric_cols),
])
```

`handle_unknown='ignore'` is the production-mindset flag: if an unseen category appears at prediction time (a new airline launches), encode it as all-zeros instead of crashing.

In [ ]:
nominal = ['airline', 'source_city', 'destination_city', 'class']
ordered = ['stops', 'departure_time', 'arrival_time']
continuous = ['duration', 'days_left']
times = ['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night']
orders = [['zero', 'one', 'two_or_more'], times, times]

> **⚠️ `remainder='drop'` is a footgun, and this is the moment to respect it.** The `ColumnTransformer` you are about to build **silently discards** every column you did not put in a group. No error. No warning.
>
> On this dataset that mistake is expensive in a very specific way: `class` is the strongest single predictor here, worth roughly **0.98 R² with it versus 0.19 without**. Forget it and your model is near-useless — and nothing in the output will say so. You will just see a bad score and start blaming your hyperparameters.
>
> So do not verify coverage by eye. Make the computer do it.

In [ ]:
assert set(df) - set(nominal + ordered + continuous) == {'flight', 'price'}

In [ ]:
prep_tree = ColumnTransformer([
    ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal),
    ('ordered', OrdinalEncoder(categories=orders, handle_unknown='use_encoded_value', unknown_value=-1), ordered),
    ('numeric', 'passthrough', continuous)
])
prep_linear = Pipeline([('encode', clone(prep_tree)), ('scale', StandardScaler())])

In [ ]:
# Deduplicate before splitting so identical fare records cannot appear in both sets.
clean = df.dropna().drop(columns='flight').drop_duplicates().reset_index(drop=True)
X = clean.drop(columns='price')
y = clean.price
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
def evaluate(name, estimator, prep):
    pipe = Pipeline([('prep', clone(prep)), ('model', estimator)])
    start = time.perf_counter()
    pipe.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    result = regression_report(y_test, pipe.predict(X_test), name)
    result['fit_seconds'] = elapsed
    record(result)
    return pipe

xgb_pipe = evaluate('XGBoost', clone(d1_model), prep_tree)
print(show_scoreboard())

### 3d — Target leakage: the expensive kind

The three leaks above were about *process*. This one is about a *feature*, and it is the one that ends projects.

Here is a feature a well-meaning teammate will propose:

```python
df['price_per_hour'] = df['price'] / df['duration']
```

It is intuitive, it is a real industry unit (yield per block hour), and adding it will take your R² to something like 0.999.

**It is computed from the target.** At prediction time you do not have `price` — predicting it is the whole job. The feature can only exist for rows where you already know the answer.

**Target leakage** is the most expensive mistake in applied ML precisely because it does not look like a bug. Nothing crashes, no warning fires, your metrics improve, and the model is worthless.

> **The test, five seconds per feature:** *"at the moment I need a prediction, do I have this value?"* If the honest answer is "no, I would need the price first" — delete it.

In [ ]:
invalid_X = X.assign(price_per_hour=y / X.duration)[['price_per_hour', 'duration', 'days_left']]
invalid_model = XGBRegressor(n_estimators=100, random_state=42, n_jobs=4)
invalid_model.fit(invalid_X.loc[X_train.index], y_train)
invalid_r2 = invalid_model.score(invalid_X.loc[X_test.index], y_test)
print('Invalid target-derived feature R²:', invalid_r2)

### 3e — One split is an anecdote

Even leak-free, our score rests on **one** arbitrary 80/20 split. Change `random_state` and the number moves. How much is measurable, and you should measure it before quoting anything.

**K-fold cross-validation** splits the data into k parts, trains k times, validates on each part once. The **mean** is a better estimate; the **standard deviation** tells you how much your single number could have been luck.

> Quote scores as **mean ± std**. A model at 0.95 ± 0.002 and one at 0.95 ± 0.04 are not the same model, and the second is not ready to ship.

In [ ]:
scores = cross_validate(xgb_pipe, X, y, cv=KFold(5, shuffle=True, random_state=42),
                        scoring=['r2', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error'],
                        return_train_score=True, n_jobs=1)
folds = pd.DataFrame({'R2': scores['test_r2'], 'MAE': -scores['test_neg_mean_absolute_error'],
                      'MAPE%': -100*scores['test_neg_mean_absolute_percentage_error'], 'train_R2': scores['train_r2']})
print(folds.agg(['mean', 'std', 'min', 'max']))

**Leak ledger**

| Configuration | R² | MAE (INR) | MAPE (%) |
|---|---:|---:|---:|
| D1 baseline (3 leaks) | 0.9782 | 1,910.67 | 14.43 |
| No counter | 0.9739 | 2,100.27 | 15.00 |
| Deduplicated | 0.9741 | 2,106.64 | 15.23 |
| XGBoost | 0.9767 | 1,975.72 | 14.31 |
| XGBoost, five folds (mean ± sample SD) | 0.97628 ± 0.00031 | 1,990.11 ± 9.35 | 14.27 ± 0.09 |

Removing the export counter caused the largest R² drop. It encodes the sorting of the export and would be unavailable for a new quote. Deduplicating removed 2,213 matching records after excluding the flight identifier and shifted the test sample slightly. The last step also changes category representation, so its score change cannot be attributed solely to training-only fitting. Five shuffled folds gave tightly grouped scores on these quotes. They do not test transfer to a later year or another market.

---
# Section 4 — Lasso vs Ridge: two ways to regularise a linear model

**Goal:** understand what regularisation *is*, what the L1 and L2 penalties do differently, and why that difference is useful rather than academic.

### Why fit a linear model at all, when you already have XGBoost?

Three reasons, and none of them is "it might win":

1. **It is a reference point.** "XGBoost got R² 0.95" is meaningless until you know a linear model gets 0.90 — or 0.55.
2. **It is interpretable.** A linear model hands you one coefficient per feature: *this airline adds ₹1,800 to the fare, all else equal.* No tree ensemble gives you a sentence like that, and that sentence is often what a client actually wants.
3. **Lasso does feature selection for free**, which is the real payoff in this section.

### The shared objective

Both models minimise squared error **plus** a penalty on the size of the coefficients:

```
Ridge  (L2):   minimise   ||y - Xw||²  +  α · Σ wⱼ²
Lasso  (L1):   minimise   ||y - Xw||²  +  α · Σ |wⱼ|
```

The only difference is `wⱼ²` versus `|wⱼ|`. That one change produces completely different behaviour:

| | **Ridge (L2)** | **Lasso (L1)** |
|---|---|---|
| Penalty | sum of **squared** coefficients | sum of **absolute** coefficients |
| Shrinks coefficients | yes, smoothly toward zero | yes |
| Sets them to **exactly zero** | essentially never | **yes** — that is the point |
| Correlated features | **shares** the weight between them | **picks one**, zeroes the rest |
| Gives you | a stable model using every feature | a **sparse** model, i.e. feature selection |
| Closed-form solution | yes | no — needs iterative optimisation |
| Reach for it when | you believe most features matter a little | you believe few features matter a lot |

### Why does L1 produce exact zeros and L2 does not?

The geometric picture is the one worth remembering. Think of the penalty as a budget constraining where `w` can live:

- The **L2** budget `Σwⱼ² ≤ t` is a **circle** (sphere). Smooth, no corners.
- The **L1** budget `Σ|wⱼ| ≤ t` is a **diamond**, and its corners sit **on the axes**.

The solution lands where the error contours first touch that region. A diamond's corners stick out, so contours tend to touch at a corner — and a corner *is* a point where some coefficient equals exactly zero. A circle has no corners, so the touch point almost always has every coefficient slightly non-zero.

> **⚠️ Scaling is mandatory here, not optional.** Both penalties sum over coefficient *magnitudes*, and a coefficient's magnitude depends on its feature's units. `duration` in hours (0–50) and `days_left` in days (1–49) are comparable, but if one were in minutes its coefficient would be ~60× smaller and the penalty would barely touch it. **Unscaled features mean the penalty silently punishes some features more than others.** This is why `prep_linear` has a `StandardScaler` and `prep_tree` does not — and why both live inside a `Pipeline`, so the scaler is fitted per fold.

In [ ]:
ols = evaluate('OLS', LinearRegression(), prep_linear)
print(show_scoreboard().loc['OLS'])

### 4a — What `alpha` actually does

`alpha` is the **regularisation strength** — how much you charge the model for large coefficients.

- `alpha → 0`: no penalty. The model fits the training data as hard as it can. **High variance**, overfits.
- `alpha → ∞`: the penalty dominates. All coefficients are crushed toward zero and the model predicts roughly a constant. **High bias**, underfits.

Somewhere in between is the right value, and you find it by validation, not by taste. This is the **bias–variance tradeoff** with a dial on it.

In [ ]:
# Pick alpha using only an inner split of the training data.
fit_x, val_x, fit_y, val_y = train_test_split(X_train, y_train, test_size=.2, random_state=42)
encoder = clone(prep_linear).fit(fit_x)
design, validation = encoder.transform(fit_x), encoder.transform(val_x)
rows = []
families = {'Ridge': Ridge(), 'Lasso': Lasso(max_iter=10000, precompute=True)}
for name, estimator in families.items():
    for alpha in [.1, 1, 10, 100, 1000]:
        model = clone(estimator).set_params(alpha=alpha).fit(design, fit_y)
        rows.append((name, alpha, model.score(design, fit_y), model.score(validation, val_y)))
sweep = pd.DataFrame(rows, columns=['model','alpha','train','validation'])
best = sweep.loc[sweep.groupby('model').validation.idxmax()].set_index('model')
for name, group in sweep.groupby('model'):
    for metric, style in [('train','--'), ('validation','-')]:
        plt.semilogx(group.alpha, group[metric], style, label=f'{name} {metric}')
plt.legend()
plt.xlabel('Penalty strength')
plt.ylabel('R²')
selected = {name: evaluate(name, clone(est).set_params(alpha=best.loc[name,'alpha']), prep_linear)
            for name, est in families.items()}
print(best)

In [ ]:
sparse_model = evaluate('Lasso (alpha 100)', Lasso(alpha=100, max_iter=10000, precompute=True), prep_linear)
def coefficients(pipe):
    return pd.Series(pipe['model'].coef_, index=pipe['prep'].get_feature_names_out())
for pipe in [selected['Lasso'], sparse_model, selected['Ridge']]:
    weights = coefficients(pipe)
    print('Kept:', weights.ne(0).sum(), 'zeroed:', weights.eq(0).sum(), weights[weights.eq(0)].index.tolist())

In [ ]:
# Optional ElasticNet extension omitted. The two required penalties are compared above.

**Linear models.** Ridge and Lasso both favored the smallest tested alpha, 0.1, on the inner validation split. Lasso had marginally lower holdout MAE, while OLS and both penalized fits rounded to the same R² of 0.9092. There are many observations for a small feature space, leaving little variance for shrinkage to remove. At alpha 0.1 Lasso zeroed two features, and at alpha 100 it zeroed eleven, including the Indigo indicator and departure time. A zero coefficient for one dummy can reflect redundancy among encoded categories. Ridge is a sensible explanatory baseline, with the trade-off that one lead-time slope misses the booking curve.

---
# Section 5 — Loss functions: what are you actually asking the model to minimise?

**Goal:** understand that "train a model" means "minimise a specific definition of wrong", that you *choose* that definition, and that the default is not always right.

Every model in Sections 3 and 4 minimised **squared error**, because that is the default and nobody told you otherwise. That was a choice, made silently, and it had consequences you can measure.

### Loss vs metric — keep these straight

| | **Loss function** | **Evaluation metric** |
|---|---|---|
| Used | *during* training, by the optimiser | *after* training, by you |
| Must be | differentiable (for gradient methods) | anything computable |
| Purpose | tells the model which direction to move | tells you how good the result is |
| Example | `reg:squarederror` | R², MAPE |

They do not have to match — but if your loss and your metric disagree about what "wrong" means, you are optimising for one thing and grading on another. That is worth doing deliberately, never by accident.

### The four losses that matter for regression

Let the error be `e = ŷ - y`.

| Loss | Formula | Penalises | Predicts the | Outlier behaviour |
|---|---|---|---|---|
| **Squared / L2 (MSE)** | `e²` | large errors **quadratically** | conditional **mean** | very sensitive — one bad row can dominate |
| **Absolute / L1 (MAE)** | `|e|` | errors **linearly** | conditional **median** | robust |
| **Huber** | `e²` for small `|e|`, linear beyond `δ` | both, by region | something between | robust, and smooth at zero |
| **Quantile / pinball** | `α·e` if `e<0`, `(α−1)·e` if `e>0` | over- and under-prediction **differently** | the **α-th quantile** | tunable, asymmetric |

Two consequences worth internalising:

- **MSE predicts the mean; MAE predicts the median.** That is not a detail. On the right-skewed price distribution you plotted in Section 2a, the mean sits well above the median, so these two losses produce *systematically different* predictions on the same data.
- **Huber exists because MSE and MAE each have one flaw.** MSE overreacts to outliers; MAE has a kink at zero that makes optimisation less stable and treats a ₹1 error with the same gradient as a ₹10,000 one. Huber is quadratic near zero and linear in the tails — and `δ` is where it switches.

> ### ⚠️ ReLU is an activation function, not a loss function
>
> This confusion is extremely common and worth clearing up precisely, because the two things live in different places in a model.
>
> **ReLU** is `f(x) = max(0, x)`. It is an **activation function**: it sits *inside* a neural network, transforming one layer's output before it reaches the next. It takes one number and returns one number. It has no idea what the true label is, so **it cannot be a loss** — a loss must compare a prediction to a truth.
>
> ```
> input -> [linear layer] -> ReLU -> [linear layer] -> ReLU -> output -> LOSS(output, truth)
>                             ^^^^                     ^^^^              ^^^^
>                        activations: shape the model   |          loss: scores the model
> ```
>
> **But `max(0, ·)` genuinely does appear inside loss functions**, which is probably where the confusion comes from — and this part is useful:
>
> - **Hinge loss** (classification, SVMs): `max(0, 1 − y·ŷ)`. Zero penalty once you are correct by a margin, linear penalty otherwise. That is literally a ReLU applied to the margin.
> - **Pinball / quantile loss** (regression, below): `max(α·e, (α−1)·e)` — two ReLU-shaped arms joined at zero, with **different slopes** on each side. The asymmetry of those slopes is the whole mechanism.
>
> So: ReLU is a **building block** of asymmetric and margin-based losses, not a loss itself. If a teammate says "let's use ReLU as our loss", the useful reply is "do you mean a hinge or a quantile loss?" — because that is almost always the idea underneath.

### 5a — Quantile loss, and why it is the interesting one for pricing

Every loss above is **symmetric**: over-predicting by ₹500 costs exactly what under-predicting by ₹500 costs.

**Pricing is not symmetric.** Consider the two ways to be wrong:

- **Quote too low.** You sell the seat, but you left money on the table — a loss of margin on one booking.
- **Quote too high.** The customer books a competitor. You lose the entire sale, and possibly the customer.

Those costs are not equal, and which is worse depends on the business situation — a nearly-full flight versus one departing half-empty. A symmetric loss cannot express any of that. **Quantile loss can**, because `α` sets the asymmetry:

- `α = 0.5` → symmetric → predicts the **median** (this is exactly MAE)
- `α = 0.9` → under-prediction punished 9× harder → predicts the **90th percentile**, a deliberately high quote
- `α = 0.1` → over-prediction punished 9× harder → predicts the **10th percentile**, a deliberately low quote

Fit three of them and you get a **prediction interval** rather than a single number — "this fare is most likely ₹6,400, and we are fairly confident it is between ₹4,900 and ₹8,800". For a pricing recommendation, that interval is far more useful than a point estimate, because it tells the user how much to trust it.

In [ ]:
evaluate('Absolute', clone(d1_model).set_params(objective='reg:absoluteerror'), prep_tree)
# sklearn sets the Huber transition using the residual distribution at each boosting stage.
evaluate('Huber', GradientBoostingRegressor(loss='huber', alpha=.9, n_estimators=150,
         learning_rate=.1, max_depth=4, subsample=.4, random_state=42), prep_tree)
print(show_scoreboard().loc[['XGBoost', 'Absolute', 'Huber']])

In [ ]:
quantiles = {}
for q in [.1, .5, .9]:
    model = Pipeline([('prep', clone(prep_tree)), ('model', clone(d1_model).set_params(
        objective='reg:quantileerror', quantile_alpha=q))])
    model.fit(X_train, y_train)
    quantiles[q] = model.predict(X_test)
    print(q, mean_pinball_loss(y_test, quantiles[q], alpha=q))
interval = pd.DataFrame(quantiles, index=y_test.index)
coverage = y_test.between(interval[.1], interval[.9]).mean()
print('80% interval coverage:', coverage)
subset = interval.sample(40, random_state=42).sort_values(.5)
plt.fill_between(range(len(subset)), subset[.1], subset[.9], alpha=.3)
plt.plot(subset[.5].to_numpy(), label='Predicted median')
plt.scatter(range(len(subset)), y_test.loc[subset.index], label='Observed', s=12)
plt.ylabel('Fare (INR)')
plt.legend()

**Loss comparison**

| Objective | MAE (INR) | RMSE (INR) | R² | MAPE (%) |
|---|---:|---:|---:|---:|
| XGBoost | 1,975.72 | 3,461.01 | 0.9767 | 14.31 |
| Absolute | 2,061.77 | 4,074.85 | 0.9677 | 13.42 |
| Huber | 2,629.71 | 4,593.41 | 0.9589 | 17.89 |

The squared-error XGBoost fit scored lower on RMSE than the absolute-error fit, while absolute error reduced percentage error. Across all three fits, the lowest MAE came from XGBoost and the lowest RMSE from XGBoost. Huber used a different boosting implementation, so that comparison also includes training differences. For a typical-fare benchmark I would optimize absolute error and report MAE alongside cabin-level percentage errors. The 10th–90th percentile band covered 81.71% of held-out fares, slightly wider in aggregate than the 80% target. If quoting too low is assumed to cost more because a traveler budgets below the eventual fare, a 0.75 quantile would be a candidate to validate with stakeholders.

---
# Section 6 — XGBoost vs Random Forest vs ARIMA

**Goal:** choose a model family on the basis of what the problem is, not what the leaderboard says.

These three are not three contestants in one race. **Two of them answer the same question in different ways, and the third answers a different question entirely** — recognising that is the actual skill in this section.

| Model | Family | What it needs | What it predicts |
|---|---|---|---|
| **Random Forest** | bagged trees | rows of features, any order | one target value **per row** |
| **XGBoost** | boosted trees | rows of features, any order | one target value **per row** |
| **ARIMA** | time series | one **ordered sequence**, equally spaced | the **next values** of that sequence |

Random Forest and XGBoost are interchangeable in their interface: both take `X, y` and predict a price for an individual ticket. ARIMA cannot do that at all — it does not take features. It takes a *sequence* and extends it.

---

### 6a — Bagging vs boosting: how RF and XGBoost actually differ

Both build many decision trees. **How** they build them is close to opposite.

| | **Random Forest (bagging)** | **XGBoost (boosting)** |
|---|---|---|
| Trees are built | **independently**, all in parallel | **sequentially**, each one after the last |
| Each tree is trained on | a bootstrap sample + a random feature subset | the **errors (gradients)** the ensemble still has |
| Individual trees are | deep, overfit, **decorrelated** | shallow, weak learners |
| Combined by | **averaging** | **adding** (scaled by the learning rate) |
| Mainly reduces | **variance** | **bias** |
| More trees → | never really worse, just slower | **can overfit**; needs `learning_rate` / early stopping |
| Tuning burden | low — works well out of the box | higher — more knobs that interact |
| Parallelises | trivially (trees are independent) | within a tree, not across them |

The intuition: **bagging** builds many different experts and averages away their individual mistakes. **Boosting** builds one expert, finds what it gets wrong, and hires a specialist for exactly that — repeatedly.

This explains their failure modes. A Random Forest rarely blows up but often plateaus slightly below a well-tuned booster. XGBoost usually wins *after tuning* and overfits cheerfully if you leave `n_estimators` high and `learning_rate` high.

In [ ]:
evaluate('Dummy', DummyRegressor(), prep_tree)
evaluate('Random Forest', RandomForestRegressor(n_estimators=100, max_depth=20,
         min_samples_leaf=2, n_jobs=4, random_state=42), prep_tree)
print(show_scoreboard().loc[['Dummy', 'Random Forest', 'XGBoost']])

**Tree ensembles.** The forest reduced ticket MAE by ₹691 relative to XGBoost. It took 29.4 seconds to fit versus 2.4 seconds for XGBoost on this machine. Against the mean-fare predictor, the MAE reductions were ₹18,495 for the forest and ₹17,804 for XGBoost. The forest is a reasonable first handoff for batch work because a teammate can start with depth and leaf size. XGBoost is attractive when frequent retraining matters.

### 6b — ARIMA: the right tool for a question we barely have

**ARIMA** = **A**uto**R**egressive **I**ntegrated **M**oving **A**verage, written `ARIMA(p, d, q)`:

- **AR(p)** — regress the value on its own previous `p` values. "Today looks like yesterday."
- **I(d)** — **difference** the series `d` times to make it **stationary** (constant mean/variance over time). A series that trends upward forever cannot be modelled directly; its *changes* usually can.
- **MA(q)** — regress on the previous `q` forecast *errors*. Lets recent shocks decay.

ARIMA is the standard tool for forecasting a single quantity through time: monthly revenue, daily passengers, weekly bookings.

### So does it apply to this dataset?

Be honest about this, because it is the real lesson:

**Mostly, no — and that is a legitimate finding.** ARIMA needs a series indexed by time, and this dataset **has no dates**. There is no departure date, no booking date, no timestamp anywhere. Without a calendar axis there is no seasonality, no trend over real time, no day-of-week effect. The dataset is a cross-section: 300,000 independent snapshots with no temporal order among them.

**But there is one real sequence in here**, and you plotted it in Section 2c: **mean price by `days_left`**. Days-to-departure is an ordered, equally-spaced index, so the booking curve is a genuine series of ~49 points — and forecasting the booking curve is a real revenue-management task. So we apply ARIMA to the thing it can actually model, and then we are precise about what we did and did not learn.

> **The transferable lesson:** your first model-selection question is never "which model scores best?" It is **"what shape is my data, and what shape does this model require?"** A model that needs a time index will not acquire one because you want it to. Picking the wrong family is a mistake no amount of hyperparameter tuning can repair.

In [ ]:
from statsmodels.tsa.stattools import adfuller
series = clean.groupby('days_left').price.mean().sort_index(ascending=False).reset_index(drop=True)
history, tail = series.iloc[:-9], series.iloc[-9:]
p_raw = adfuller(history)[1]
p_diff = adfuller(history.diff().dropna())[1]
print('Training ADF p-values, level and first difference:', p_raw, p_diff)
fig, axes = plt.subplots(1, 2)
history.plot(ax=axes[0], title='Training booking curve')
history.diff().plot(ax=axes[1], title='First difference')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
d = int(p_raw >= .05)
start = time.perf_counter()
fits = [ARIMA(history, order=(p,d,q)).fit() for p,q in [(1,0),(0,1),(2,2)]]
for fit in fits:
    print(fit.model.order, fit.aic, 'converged:', fit.mle_retvals['converged'])
arima = min((fit for fit in fits if fit.mle_retvals['converged']), key=lambda fit: fit.aic)
arima_seconds = time.perf_counter() - start
forecast = arima.forecast(len(tail))
naive = pd.Series(history.iloc[-1], index=tail.index)
arima_mae = mean_absolute_error(tail, forecast)
naive_mae = mean_absolute_error(tail, naive)
print(arima.model.order, arima_mae, naive_mae)
for line, label in [(history,'Train'), (tail,'Held out'), (forecast,'ARIMA'), (naive,'Last value')]:
    line.plot(label=label)
plt.legend()
plt.ylabel('Mean fare (INR)')

**Booking-curve forecast.** On the last nine lead times, ARIMA(1,1,0) had MAE ₹1,626.03 and the last-value forecast had MAE ₹1,647.26. A ₹21.23 advantage on nine points is weak evidence for repeatable improvement. ADF failed to reject a unit root on the training levels and on their first differences, so one difference is a working assumption with limited support. The series summarizes average fare as departure approaches. Aggregation removes the carrier, route, cabin, and stops needed to price an individual itinerary. Use it as a descriptive booking-curve benchmark only.

> **For the curious: SARIMAX.** The `statsmodels` family does extend to seasonality and features: `SARIMAX` adds a seasonal component and an `exog=` argument for external regressors. With real departure dates you could model weekly and holiday seasonality and feed in fuel price or capacity — which is much closer to how an airline forecasts demand. We cannot, because this dataset has no dates. Worth knowing the door exists.

### 6c — Model selection

| Model | MAE (INR) | R² | Fit time (s) | Interpretable? | Use |
|---|---:|---:|---:|---|---|
| Dummy | 19,779.74 | -0.0000 | 0.36 | Yes | Mean-fare reference |
| Ridge | 4,518.26 | 0.9092 | 0.27 | Yes | Signed effects, stable baseline |
| Lasso | 4,518.21 | 0.9092 | 0.30 | Yes | Sparse explanation |
| Random Forest | 1,284.76 | 0.9852 | 29.40 | Limited | Batch, nonlinear cabin interactions |
| XGBoost | 1,975.72 | 0.9767 | 2.41 | Limited | Fast boosted-tree refits |
| ARIMA | 1,626.03 (curve) | — | 0.05 | Partly | Aggregate booking curve |

Random Forest is the candidate for a batch fare benchmark. Its splits can represent the sharp rise near departure, its trees can learn cabin-specific responses without hand-built interactions, and bagging provides a straightforward training recipe for teammates to repeat. The main cost is slower fitting and a larger collection of trees to maintain. A chronological, market-specific evaluation would be necessary before operational use.

---
# Section 7 — Defend your choice

### 7a — The full scoreboard

In [ ]:
print(show_scoreboard().sort_values('MAE'))

### 7b — What this means for American Airlines

We are modelling **Indian domestic fares scraped from EaseMyTrip in 2022**. American Airlines is a different carrier in a different market in a different year. Be precise about what transfers.

**What transfers — the structure of the problem and the method.** Airline revenue management has the same shape everywhere: fares move with the booking horizon, cabin class dominates price, route density and competition set the base level, and time-of-day and connections differentiate the product. Your model-selection reasoning transfers completely: tree ensembles for per-ticket prediction, asymmetric loss where over- and under-pricing cost differently, a time-series model for forecasting an aggregate curve through real time.

**What does not transfer:** nearly all the specifics. Different regulatory regime, competitive set, cost base, currency, and demand seasonality — and 2022 was still distorted by post-pandemic recovery.

**What the dataset is missing that real pricing requires:**

| Missing | Why it matters |
|---|---|
| **Seat inventory / load factor** | Revenue management is fundamentally about *remaining seats*. We have no capacity data at all — the largest single gap. |
| **Departure dates** | We have `days_left` but not *when*. No seasonality, no holidays, no day-of-week. **This is also precisely why ARIMA had so little to work with.** |
| **Competitor fares** | Fares are set against what everyone else charges on the route, in near-real-time. |
| **Fare class / fare basis** | "Economy" spans a dozen booking classes with different prices and restrictions; our `class` flattens all of it to two values. |
| **Demand signals** | Searches, booking pace, events — the leading indicators pricing teams actually watch. |

### 7c — Reflection

Answer each in the cell below it, in your own words. A PM will ask these in your PR review, so treat it as practice for that conversation.

---

**Q1 — The honest number.** You produced several R² values for the same task. List them with one line each on what changed, then name the single number you would put in front of American Airlines and defend it in two sentences.

**Answer.** The row-level R² moved from 0.9782 with the export counter to 0.9739 without it, 0.9741 after deduplication, and 0.9767 in the clean pipeline. I would quote the five-fold XGBoost R² of 0.97628 ± 0.00031 for this practice market. It summarizes variation across multiple random splits, while future dates and AA fares remain untested.

**Q2 — Model family.** You would ship one of Ridge / Lasso / Random Forest / XGBoost. Name it, and give **three** reasons that are not "it had the best score" — e.g. fit time, tuning burden, interpretability, robustness, what the stakeholder needs. Then name the strongest argument *against* your choice.

**Answer.** Random Forest is the candidate for a batch fare benchmark. Its splits can represent the sharp rise near departure, its trees can learn cabin-specific responses without hand-built interactions, and bagging provides a straightforward training recipe for teammates to repeat. The main cost is slower fitting and a larger collection of trees to maintain. A chronological, market-specific evaluation would be necessary before operational use.

**Q3 — Lasso vs Ridge.** Explain to a teammate who has not done this deliverable why L1 produces exact zeros and L2 does not. Then say which you would use on this dataset and why — including the possibility that neither penalty mattered much here, and what about this dataset makes that the case.

**Answer.** The L1 penalty has a corner at zero, so a range of small data gradients can be balanced at an exactly zero coefficient. The smooth L2 gradient goes to zero there and generally shrinks coefficients continuously. I would use Ridge for a stable baseline with correlated indicators. In this large sample the tested penalties changed holdout accuracy very little.

**Q4 — Loss functions.** Your Deliverable 1 model minimised squared error because that is the default. Having seen the comparison: was that the right default for this data? Which loss would you train on now, and which metric would you report? If those two differ, justify the mismatch.

**Answer.** The squared-error XGBoost fit scored lower on RMSE than the absolute-error fit, while absolute error reduced percentage error. Across all three fits, the lowest MAE came from XGBoost and the lowest RMSE from XGBoost. Huber used a different boosting implementation, so that comparison also includes training differences. For a typical-fare benchmark I would optimize absolute error and report MAE alongside cabin-level percentage errors. The 10th–90th percentile band covered 81.71% of held-out fares, slightly wider in aggregate than the 80% target. If quoting too low is assumed to cost more because a traveler budgets below the eventual fare, a 0.75 quantile would be a candidate to validate with stakeholders.

**Q5 — ARIMA and problem shape.** Someone proposes ARIMA for predicting individual ticket prices. Explain in three sentences why that does not work, and what they would need to change about the data or the question for a time-series model to be the right call.

**Answer.** On the last nine lead times, ARIMA(1,1,0) had MAE ₹1,626.03 and the last-value forecast had MAE ₹1,647.26. A ₹21.23 advantage on nine points is weak evidence for repeatable improvement. ADF failed to reject a unit root on the training levels and on their first differences, so one difference is a working assumption with limited support. The series summarizes average fare as departure approaches. Aggregation removes the carrier, route, cabin, and stops needed to price an individual itinerary. Use it as a descriptive booking-curve benchmark only.

**Q6 — Leakage.** Explain leakage to a teammate using an example from your own notebook run (not one from the text). Then name the one habit from this notebook you will carry into Deliverable 3 to prevent it.

**Answer.** Price divided by duration raised R² to 0.99955 in the deliberate leakage test. Computing that feature requires the fare we are trying to estimate. For each future feature, record what raw inputs it requires and when those inputs become available.

**Q7 — Team git.** Deliverable 1 was `deliverable1/<your-name>` → `main`. Deliverable 2 is `team-##/<your-name>` → `team-##/main` → `main`. What does the extra layer buy a project with 70+ contributors? Then describe something that actually went wrong or felt awkward in your team's git this week — a real friction point is worth more here than a clean answer.

**Answer.** Team branches give peers a place to integrate and review individual results before the PM reviews one team submission. During setup, the existing remote team-13 branch prevented the member name team-13/Rahul on GitHub. The chosen member branch is team-13-Rahul. Publishing and the assigned peer review are still pending.

---
## Before you submit — checklist

**The notebook**

1. **Kernel → Restart & Run All.** It must run top-to-bottom with no errors. A notebook that only works in the order you happened to click cells is not reproducible.
2. Every `# TODO` cell is filled in.
3. Every markdown answer prompt has a real answer. "*(Write here)*" left anywhere is an incomplete deliverable.
4. The three summary tables are filled in: the EDA decisions (2e), the model-selection table (6c), and the scoreboard (7a).
5. **Kernel → Restart & Clear Output** before committing. Keeps the diff reviewable and prevents most notebook merge conflicts.

**Git hygiene**

6. `git status` shows **no** `.csv`. If it does, stop — do not commit, and check [`.gitignore`](../.gitignore).
7. You are on your own member branch — not your team's trunk, and definitely not `main`. Check with `git branch --show-current`.

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push
```

8. Open a PR from your member branch into **your team's trunk branch**. GitHub defaults the base to `main` — change it by hand.
9. Request review from the teammate assigned to you in [`README.md`](README.md), and **review the PR assigned to you**. Reviewing is half the job.
10. Once every member's PR has merged into the team trunk, the team fills in [`TEAM_REPORT.md`](TEAM_REPORT.md) together and a PM reviews the trunk → `main`.

---

You arrived with one model and one score. You are leaving with five models, four loss functions, an honest number, and — most importantly — a reason for every choice.

Deliverable 3 moves to the approved American Airlines data.